In [1]:
import pandas as pd
import numpy as np
import re
import plotly.express as px
import reverse_geocoder as rg # type: ignore
# -------------------------------------------------
from sklearn.linear_model import LinearRegression

In [2]:
data = pd.read_excel("Crime_Incidents_in_the_Last_30_Days.xlsx")
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

In [3]:
data['DISTRICT'] = data['DISTRICT'].convert_dtypes(int)
data['PSA'] = data['PSA'].convert_dtypes(int)
data = data.drop(columns = ['XBLOCK','YBLOCK','CENSUS_TRACT','WARD'])
data = data.rename(columns = {'BLOCK_GROUP':'BLOCK_GROUP-CENSUS_TRACT & WARD'})
data['BLOCK_GROUP-CENSUS_TRACT & WARD'] = data['BLOCK_GROUP-CENSUS_TRACT & WARD'].astype(str).str.lstrip('0')
data = data.where(data.notna(), None)
  
data['START_DATE'] = pd.to_datetime(data['START_DATE'])
data['END_DATE'] = pd.to_datetime(data['END_DATE'])
data['REPORT_DAT'] = pd.to_datetime(data['REPORT_DAT'])

data['NEIGHBORHOOD_CLUSTER'] = pd.Categorical(data['NEIGHBORHOOD_CLUSTER'],categories=sorted(data['NEIGHBORHOOD_CLUSTER'].dropna().unique(), key=lambda x: int(re.search(r'\d+', x).group())),ordered=True)
data = data.sort_values('NEIGHBORHOOD_CLUSTER')
  
data['incident_duration'] = ((data['END_DATE'] - data['START_DATE']).dt.total_seconds() / 60).abs()
data['reporting_gap'] = ((data['REPORT_DAT'] - data['END_DATE']).dt.total_seconds() / 60).abs()
  
coords = list(zip(data['LATITUDE'], data['LONGITUDE']))   #type: ignore
results = rg.search(coords)
data['area_name'] = [r['name'] for r in results]   #type: ignore
  
data.columns = data.columns.str.upper()

Loading formatted geocoded file...


In [ ]:
data.head(20)
# data.info()
# data.describe()
# data.groupby(['NEIGHBORHOOD_CLUSTER','PSA','DISTRICT','AREA_NAME'])['BID'].value_counts()

,CCN,SHIFT,METHOD,OFFENSE,BLOCK,XBLOCK,YBLOCK,WARD,ANC,DISTRICT,PSA,NEIGHBORHOOD_CLUSTER,BLOCK_GROUP,CENSUS_TRACT,VOTING_PRECINCT,LATITUDE,LONGITUDE,BID,START_DATE,END_DATE,REPORT_DAT
0,25161987,MIDNIGHT,OTHERS,THEFT/OTHER,3100 - 3299 BLOCK OF 14TH STREET NW,397162.060000,140182.430000,1,1A,3.0,302.0,Cluster 2,002802 1,2802,Precinct 39,38.929517,-77.032730,NaN,10/25/2025 7:55 AM,10/25/2025 1:55 AM,10/27/2025 6:00 AM
1,25162437,EVENING,OTHERS,THEFT/OTHER,3100 - 3299 BLOCK OF 14TH STREET NW,397162.060000,140182.430000,1,1A,3.0,302.0,Cluster 2,002802 1,2802,Precinct 39,38.929517,-77.032730,NaN,10/18/2025 6:00 AM,10/18/2025 3:07 AM,10/19/2025 2:52 AM
2,25157565,DAY,GUN,ROBBERY,1400 - 1599 BLOCK OF NEWTON STREET NW,396999.960000,140521.120000,1,1A,4.0,408.0,Cluster 2,002801 1,2801,Precinct 41,38.932567,-77.034601,NaN,10/18/2025 9:00 PM,2025-10-18 12:03:00,10/23/2025 8:56 AM
3,25160412,EVENING,OTHERS,THEFT/OTHER,2818 - 2899 BLOCK OF SHERMAN AVENUE NW,397754.160000,139859.270000,1,1A,3.0,304.0,Cluster 2,003500 1,3500,Precinct 37,38.926607,-77.025900,NaN,10/21/2025 10:19 PM,10/21/2025 11:04 PM,10/22/2025 3:05 AM
4,25163492,DAY,OTHERS,THEFT/OTHER,3100 - 3299 BLOCK OF 14TH STREET NW,397162.060000,140182.430000,1,1A,3.0,302.0,Cluster 2,002802 1,2802,Precinct 39,38.929517,-77.032730,NaN,10/27/2025 6:32 PM,10/27/2025 7:30 PM,10/27/2025 7:43 AM
5,25162313,DAY,GUN,HOMICIDE,3200 - 3299 BLOCK OF HIATT PLACE NW,396986.370000,140220.860000,1,1A,3.0,302.0,Cluster 2,002802 1,2802,Precinct 39,38.929862,-77.034756,NaN,11/3/2025 7:45 PM,11/3/2025 11:40 AM,11/4/2025 8:04 AM
6,25162989,DAY,OTHERS,THEFT/OTHER,3100 - 3199 BLOCK OF 13TH STREET NW,397424.080000,140141.130000,1,1A,3.0,302.0,Cluster 2,003000 1,3000,Precinct 39,38.929146,-77.029708,NaN,10/26/2025 5:27 PM,10/26/2025 5:45 PM,10/26/2025 6:06 AM
7,25163995,DAY,OTHERS,THEFT/OTHER,3100 - 3299 BLOCK OF 14TH STREET NW,397162.060000,140182.430000,1,1A,3.0,302.0,Cluster 2,002802 1,2802,Precinct 39,38.929517,-77.032730,NaN,10/28/2025 6:15 PM,10/28/2025 7:00 PM,10/28/2025 7:43 AM
8,25156355,EVENING,OTHERS,THEFT/OTHER,3100 - 3299 BLOCK OF 14TH STREET NW,397162.060000,140182.430000,1,1A,3.0,302.0,Cluster 2,002802 1,2802,Precinct 39,38.929517,-77.032730,NaN,10/13/2025 10:32 PM,10/13/2025 11:22 PM,10/13/2025 11:22 AM
9,25160750,DAY,OTHERS,THEFT F/AUTO,1400 - 1509 BLOCK OF MERIDIAN PLACE NW,397029.060000,140633.180000,1,1A,4.0,408.0,Cluster 2,002801 1,2801,Precinct 41,38.933577,-77.034266,NaN,10/22/2025 8:00 AM,10/22/2025 9:00 AM,10/22/2025 4:45 AM


In [ ]:

fig1 = px.box(data,x ='METHOD',y='OFFENSE',color='OFFENSE',template='plotly_dark')
fig1.show()

fig2 = px.area(data.groupby(['OFFENSE','NEIGHBORHOOD_CLUSTER','BLOCK_GROUP-CENSUS_TRACT & WARD','AREA_NAME'])['DISTRICT'],template='plotly_dark')
fig2.show()

fig3 = px.box(data,x='NEIGHBORHOOD_CLUSTER',y='AREA_NAME',template='plotly_dark')
fig3.show()

fig4 = px.pie(data,names='OFFENSE',values=data['INCIDENT_DURATION'].apply(convert_to_hours_as_nums),template='plotly_dark') # type: ignore
fig4.show()

fig5 = px.treemap(data,path=['OFFENSE','METHOD','SHIFT'],template='plotly_dark')
fig5.show()

fig6 = px.area(data, x='INCIDENT_DURATION', y='REPORTING_GAP',template='plotly_dark')
fig6.show()




C:\Users\HP\AppData\Local\Temp\ipykernel_10776\201085591.py:4: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [7]:
  
def format_duration(mins) :
    if mins < 0 :
        return None
    hours = int(mins // 60)
    remaining_mins = int(mins % 60)
    if hours == 0 :
        return f"{remaining_mins}M"
    elif remaining_mins == 0 :
        return f"{hours}H"
    else :
        return f"{hours}H {remaining_mins}M"
    
data['INCIDENT_DURATION'] = data ['INCIDENT_DURATION'].apply(format_duration)
  
def format_duration(mins) :
    if mins < 0 :
        return None
    hours = int(mins // 60)
    remaining_mins = int(mins % 60)
    if hours == 0 :
        return f"{remaining_mins}M"
    elif remaining_mins == 0 :
        return f"{hours}H"
    else :
        return f"{hours}H {remaining_mins}M"
    
data['REPORTING_GAP'] = data ['REPORTING_GAP'].apply(format_duration)
  
def convert_to_hours_as_nums(x) :
    h = 0
    m = 0
    
    if 'H' in x :
        h = int(x.split('H')[0])
        x = x.split('H')[1]
        
    if 'M' in x :
        m = int(x.replace('M',""))
        
    return h + m/60

In [12]:
data.to_excel('crime_data_cleaned.xlsx', index=False)

In [ ]:
data['SHIFT'].value_counts()

In [ ]:
data['METHOD'].value_counts()

In [ ]:
data['OFFENSE'].value_counts()

In [ ]:
data.isnull().sum()